# Record history and change feeds
What changed, when and by whom: one record's versions, and the change document feeds (Activity Streams) that downstream systems poll to stay in sync. Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

## The change feeds
One feed for works, one for instances. The entry point says how many changes there are and where the first and last pages are; pages are oldest first.

In [ ]:
import pandas as pd

feeds = {kind: bc.get(f"/api/change_documents/{kind}/feed").json() for kind in ["works", "instances"]}
pd.DataFrame([{"feed": k, "changes": f["totalItems"], "first page": f["first"]["id"], "last page": f["last"]["id"]} for k, f in feeds.items()])

## The latest changes
The newest page of each feed (and the one before it, if the newest is short).

In [ ]:
def page(kind, n):
    return bc.get(f"/api/change_documents/{kind}/page/{n}").json()

rows = []
for kind, feed in feeds.items():
    if not feed["totalItems"]:
        continue   # an empty feed names page 0 as its last page, which the API can't serve (a known bug)
    last = int(feed["last"]["id"].rsplit("/", 1)[1])
    for n in sorted({last, max(last - 1, 1)}):
        for item in page(kind, n)["orderedItems"]:
            rows.append({"feed": kind, "page": n, "change": item["type"], "published": item["published"], "record": item["object"]["id"]})
changes = pd.DataFrame(rows, columns=["feed", "page", "change", "published", "record"]).sort_values("published", ascending=False)
changes.head(20)

In [ ]:
changes["day"] = changes["published"].str[:10]
changes.groupby(["day", "feed", "change"]).size().unstack(fill_value=0).tail(10)

## One record's versions
The most recently changed work, or set `WORK_UUID`. Compares the newest two versions.

In [ ]:
# The newest change whose work still exists (deleted ones stay in the feed)
WORK_UUID = next((u for u in (r.rsplit("/", 1)[1] for r in changes[changes["feed"] == "works"]["record"])
                  if bc.get(f"/api/works/{u}/versions").ok), None) or bc.sample("works")
versions = bc.get(f"/api/works/{WORK_UUID}/versions").json()["versions"]
print(WORK_UUID, "-", len(versions), "version(s)")
pd.DataFrame(versions)

In [ ]:
import difflib, json

if len(versions) < 2:
    print("Only one version: nothing to compare.")
else:
    old, new = (bc.get(f"/api/works/{WORK_UUID}/version/{v['id']}").json()["data"] for v in versions[-2:])
    diff = difflib.unified_diff(json.dumps(old, indent=1, sort_keys=True).splitlines(), json.dumps(new, indent=1, sort_keys=True).splitlines(),
                                f"version {versions[-2]['id']}", f"version {versions[-1]['id']}", lineterm="", n=1)
    print("\n".join(list(diff)[:120]) or "No differences in the data.")